# Notebook 04 — Visão 360º do Cliente & Respostas Poliglotas às 3 Perguntas de Negócio + Bônus de Trade Marketing

**Objetivo:** Demonstrar a **Biblioteca de Consulta Poliglota (`basketlens.queries`)** operando com *pushdown* integral aos 3 bancos (MongoDB, Apache Cassandra e Redis) e zero carregamento de CSV em memória:
1. **Visão 360º Unificada (`build_360_view(208)`) + Benchmark Cache-Aside (`GET profile:208` -> MongoDB fallback -> `SETEX`)**.
2. **Pergunta 1**: Qual o perfil completo e histórico recente de compras de um cliente (`household_key = 208`)?
3. **Pergunta 2**: Quais campanhas e cupons estão ativos **agora** (`TTL > 0`) para um domicílio (`household_key = 208`)?
4. **Pergunta 3**: Qual o gasto semanal médio por faixa demográfica (`classification_1` = Faixa Etária e `classification_3` = Nível de Renda) nos 801 domicílios mapeados?
5. **Bônus de Trade Marketing**: Qual o *lift* promocional de Display & Mailer (`72717` do PDF e `862349`) no Cassandra e qual a vitrine semanal ativa no Redis?

In [1]:
import pprint
from basketlens.ingestion import ingest_redis_state
from basketlens.queries import (
    benchmark_cache_aside_latency,
    build_360_view,
    get_active_campaigns_for_household,
    get_avg_weekly_spend_by_demographic,
    get_household_product_history,
    get_household_profile_and_recent_history,
    get_product_causal_promotion_lift,
    get_store_weekly_showcase,
)
from basketlens.schemas.redis_schema import clear_redis_namespace

# Garante que o Redis está posicionado em REFERENCE_DAY = 650 e limpa o cache de perfis
_ = ingest_redis_state(reference_day=650, seconds_per_day=5)
clear_redis_namespace(pattern="profile:*")

print("1) Primeira chamada build_360_view(208) — Cold Cache Miss (lê do MongoDB e grava SETEX no Redis):")
view_miss = build_360_view(208, limit_transactions=10)
print("Cache Metadata:", view_miss["cache"])
print("Latências (ms):", view_miss["latency_ms"])

print("\n2) Segunda chamada build_360_view(208) — Warm Cache Hit (lê direto de profile:208 no Redis):")
view_hit = build_360_view(208, limit_transactions=10)
print("Cache Metadata:", view_hit["cache"])
print("Latências (ms):", view_hit["latency_ms"])

print("\n3) Benchmark estatístico Cache-Miss vs. Cache-Hit (15 iterações):")
bench = benchmark_cache_aside_latency(household_key=208, iterations=15)
print({
    "household_key": bench["household_key"],
    "iterations": bench["iterations"],
    "avg_cache_miss_ms": bench["avg_cache_miss_ms"],
    "avg_cache_hit_ms": bench["avg_cache_hit_ms"],
    "speedup_factor": bench["speedup_factor"],
})

1) Primeira chamada build_360_view(208) — Cold Cache Miss (lê do MongoDB e grava SETEX no Redis):


Cache Metadata: {'key': 'profile:208', 'cache_hit': False, 'source': 'mongodb', 'ttl_remaining_seconds': 60}
Latências (ms): {'profile_lookup_ms': 19.002, 'cassandra_transactions_ms': 14.358, 'redis_campaigns_ms': 61.685, 'total_ms': 259.038}

2) Segunda chamada build_360_view(208) — Warm Cache Hit (lê direto de profile:208 no Redis):
Cache Metadata: {'key': 'profile:208', 'cache_hit': True, 'source': 'redis', 'ttl_remaining_seconds': 60}
Latências (ms): {'profile_lookup_ms': 4.86, 'cassandra_transactions_ms': 9.066, 'redis_campaigns_ms': 42.429, 'total_ms': 184.271}

3) Benchmark estatístico Cache-Miss vs. Cache-Hit (15 iterações):


{'household_key': 208, 'iterations': 15, 'avg_cache_miss_ms': 5.352, 'avg_cache_hit_ms': 0.704, 'speedup_factor': 7.6}


## 2. Pergunta de Negócio 1 — Perfil Demográfico/Comportamental (MongoDB) + Histórico Recente (Cassandra)

In [2]:
q1 = get_household_profile_and_recent_history(household_key=208, limit=15)
print("Demografia de John Smith (household_key = 208):")
pprint.pprint(q1["demographics"])

print("\nResumo Comportamental (MongoDB behavior_summary):")
pprint.pprint(q1["behavior_summary"])

print(f"\nÚltimas {len(q1['recent_baskets'])} cestas embutidas no documento MongoDB (recent_baskets):")
for b in q1["recent_baskets"]:
    print(f" - Basket {b['basket_id']} | Dia {b['day']} (Semana {b['week_no']}) | Loja {b['store_id']} | Total=${b['total']:.2f} | Itens={len(b['items'])}")

print(f"\nTop 5 transações mais recentes no Cassandra (ordenadas por week_no DESC, day DESC):")
for tx in q1["recent_transactions"][:5]:
    print(tx)

q1_prod = get_household_product_history(household_key=208, product_id=1017772)
print(f"\nHistórico específico do produto 1017772 para o domicílio 208 ({q1_prod['purchase_count']} compras):")
for p in q1_prod["purchases"]:
    print(p)

Demografia de John Smith (household_key = 208):
{'age_group': 'Age Group4',
 'classification_1': 'Age Group4',
 'classification_2': 'Y',
 'classification_3': 'Level5',
 'classification_4': '2',
 'classification_5': 'Group5',
 'hh_comp_group': 'Group5',
 'homeowner_desc': 'Homeowner',
 'household_size': '2',
 'income_level': 'Level5',
 'kid_category_desc': 'None/Unknown',
 'marital_code': 'Y'}

Resumo Comportamental (MongoDB behavior_summary):
{'active_weeks': 80,
 'avg_basket_value': 30.69,
 'avg_weekly_spend': 53.7,
 'first_day': 49,
 'last_day': 701,
 'top_commodities': ['COUPON/MISC ITEMS', 'BEEF', 'ICE CREAM/MILK/SHERBTS'],
 'top_departments': ['GROCERY', 'MEAT', 'KIOSK-GAS'],
 'total_baskets': 140,
 'total_spend': 4296.33,
 'updated_at': '2026-10-19T10:30:15Z'}

Últimas 5 cestas embutidas no documento MongoDB (recent_baskets):
 - Basket 42039050670 | Dia 692 (Semana 100) | Loja 327 | Total=$18.28 | Itens=5
 - Basket 42039050687 | Dia 692 (Semana 100) | Loja 327 | Total=$7.19 | Ite

## 3. Pergunta de Negócio 2 — Campanhas e Cupons Ativos Agora (`TTL > 0` no Redis)

In [3]:
# Garante TTL fresco no Redis para demonstração
_ = ingest_redis_state(reference_day=650, seconds_per_day=5)
q2 = get_active_campaigns_for_household(household_key=208)

print(f"Campanhas ativas para household_key = 208 em REFERENCE_DAY = 650 ({q2['active_campaign_count']} campanha ativa):")
for camp in q2["active_campaigns"]:
    print({
        "campaign": camp["campaign"],
        "description": camp["description"],
        "targeting_mechanism": camp["targeting_mechanism"],
        "start_day": camp["start_day"],
        "end_day": camp["end_day"],
        "ttl_seconds": camp["ttl_seconds"],
        "coupon_count": camp["coupon_count"],
    })
    print("\nAmostra de cupons elegíveis da Campanha 22 (incluindo 51800000050):")
    for c in camp["coupons"][:8]:
        print(" ", c)

Campanhas ativas para household_key = 208 em REFERENCE_DAY = 650 (1 campanha ativa):
{'campaign': 22, 'description': 'TypeB', 'targeting_mechanism': 'Triggered/segmented campaign pool', 'start_day': 624, 'end_day': 656, 'ttl_seconds': 25, 'coupon_count': 21}

Amostra de cupons elegíveis da Campanha 22 (incluindo 51800000050):
  {'coupon_upc': '10000085486', 'campaign': 22, 'product_id': 41780, 'product_count': 1343, 'ttl_seconds': 25}
  {'coupon_upc': '10000085487', 'campaign': 22, 'product_id': 45339, 'product_count': 45, 'ttl_seconds': 25}
  {'coupon_upc': '10000089316', 'campaign': 22, 'product_id': 41783, 'product_count': 13, 'ttl_seconds': 25}
  {'coupon_upc': '51312010033', 'campaign': 22, 'product_id': 73081, 'product_count': 54, 'ttl_seconds': 25}
  {'coupon_upc': '51450050050', 'campaign': 22, 'product_id': 64516, 'product_count': 24, 'ttl_seconds': 25}
  {'coupon_upc': '51800000050', 'campaign': 22, 'product_id': 72717, 'product_count': 38, 'ttl_seconds': 25}
  {'coupon_upc':

## 4. Pergunta de Negócio 3 — Gasto Semanal Médio por Faixa Demográfica (`classification_1` e `classification_3`)

In [4]:
q3_age = get_avg_weekly_spend_by_demographic("classification_1")
print(f"1) Gasto Semanal Médio por Faixa Etária (classification_1) — {q3_age['total_demographic_households']} domicílios (ALLOW FILTERING = {q3_age['allow_filtering_used']}):")
print(f"{'FAIXA (classification_1)':<26} | {'DOMICÍLIOS':<10} | {'GASTO TOTAL ($)':<16} | {'SEMANAS ATIVAS':<15} | {'GASTO SEMANAL MÉDIO ($)':<23}")
print("-" * 98)
for g in q3_age["groups"]:
    print(
        f"{g['group_value']:<26} | "
        f"{g['household_count']:<10} | "
        f"{g['total_spend']:<16.2f} | "
        f"{g['total_active_weeks']:<15} | "
        f"{g['avg_weekly_spend']:<23.2f}"
    )
print("Latências (ms):", q3_age["latency_ms"])

q3_income = get_avg_weekly_spend_by_demographic("classification_3")
print(f"\n2) Gasto Semanal Médio por Nível de Renda (classification_3) — {q3_income['total_demographic_households']} domicílios:")
print(f"{'FAIXA (classification_3)':<26} | {'DOMICÍLIOS':<10} | {'GASTO TOTAL ($)':<16} | {'SEMANAS ATIVAS':<15} | {'GASTO SEMANAL MÉDIO ($)':<23}")
print("-" * 98)
for g in q3_income["groups"]:
    print(
        f"{g['group_value']:<26} | "
        f"{g['household_count']:<10} | "
        f"{g['total_spend']:<16.2f} | "
        f"{g['total_active_weeks']:<15} | "
        f"{g['avg_weekly_spend']:<23.2f}"
    )
print("Latências (ms):", q3_income["latency_ms"])

1) Gasto Semanal Médio por Faixa Etária (classification_1) — 801 domicílios (ALLOW FILTERING = False):
FAIXA (classification_1)   | DOMICÍLIOS | GASTO TOTAL ($)  | SEMANAS ATIVAS  | GASTO SEMANAL MÉDIO ($)
--------------------------------------------------------------------------------------------------
Age Group1                 | 46         | 216583.02        | 3252            | 66.60                  
Age Group2                 | 142        | 777936.25        | 9826            | 79.17                  
Age Group3                 | 194        | 1242146.79       | 13773           | 90.19                  
Age Group4                 | 288        | 1657467.05       | 20400           | 81.25                  
Age Group5                 | 59         | 298229.02        | 4347            | 68.61                  
Age Group6                 | 72         | 305354.13        | 4890            | 62.44                  
Latências (ms): {'mongodb_projection_ms': 23.347, 'cassandra_801_partitions_m


2) Gasto Semanal Médio por Nível de Renda (classification_3) — 801 domicílios:
FAIXA (classification_3)   | DOMICÍLIOS | GASTO TOTAL ($)  | SEMANAS ATIVAS  | GASTO SEMANAL MÉDIO ($)
--------------------------------------------------------------------------------------------------
Level1                     | 61         | 339136.30        | 4331            | 78.30                  
Level2                     | 74         | 303187.20        | 5225            | 58.03                  
Level3                     | 77         | 380231.38        | 5458            | 69.66                  
Level4                     | 172        | 825945.47        | 12019           | 68.72                  
Level5                     | 192        | 1094852.10       | 13735           | 79.71                  
Level6                     | 96         | 558962.90        | 6503            | 85.95                  
Level7                     | 34         | 201543.17        | 2442            | 82.53                

## 5. Consultas Bônus de Trade Marketing — Lift Causal de Display/Mailer (`72717` e `862349`) + Vitrine Semanal no Redis

In [5]:
lift_72717 = get_product_causal_promotion_lift(product_id=72717)
week40_72717 = [w for w in lift_72717["weekly_breakdown"] if w["week_no"] == 40][0]
print("1) Estudo de Caso do PDF — Produto 72717 na Semana 40 (cruzamento sales_by_product_week + causal_promotions_by_store_week):")
pprint.pprint({
    "product_id": 72717,
    "week_no": week40_72717["week_no"],
    "total_units": week40_72717["total_units"],
    "total_sales": week40_72717["total_sales"],
    "active_display_codes": week40_72717["active_display_codes"],
    "active_mailer_codes": week40_72717["active_mailer_codes"],
    "promoted_stores_count": week40_72717["promoted_stores_count"],
})

lift_862349 = get_product_causal_promotion_lift(product_id=862349)
print("\n2) Comparativo de Lift Promocional em Produto de Alto Giro (PRODUCT_ID = 862349):")
pprint.pprint({
    "product_id": lift_862349["product_id"],
    "weeks_with_sales": lift_862349["weeks_with_sales"],
    "weeks_with_causal_promotions": lift_862349["weeks_with_causal_promotions"],
    "avg_weekly_sales_promoted": lift_862349["avg_weekly_sales_promoted"],
    "avg_weekly_sales_baseline": lift_862349["avg_weekly_sales_baseline"],
    "promotional_lift_pct": lift_862349["promotional_lift_pct"],
})

showcase = get_store_weekly_showcase(store_id=288, week_no=94, top_n=10)
print("\n3) Vitrine Semanal de Trade Marketing no Redis (Semana 94, Loja 288):")
print("Top 5 Commodities (ZSET ranking:week:94):", showcase["top_commodities"][:5])
print("Total de produtos em display físico na loja 288 (SET store:288:week:94:displays):", showcase["displayed_products_count"])

1) Estudo de Caso do PDF — Produto 72717 na Semana 40 (cruzamento sales_by_product_week + causal_promotions_by_store_week):
{'active_display_codes': ['2'],
 'active_mailer_codes': ['A'],
 'product_id': 72717,
 'promoted_stores_count': 11,
 'total_sales': 2.5,
 'total_units': 2,
 'week_no': 40}



2) Comparativo de Lift Promocional em Produto de Alto Giro (PRODUCT_ID = 862349):
{'avg_weekly_sales_baseline': 19.43,
 'avg_weekly_sales_promoted': 86.96,
 'product_id': 862349,
 'promotional_lift_pct': 347.56,
 'weeks_with_causal_promotions': 93,
 'weeks_with_sales': 101}

3) Vitrine Semanal de Trade Marketing no Redis (Semana 94, Loja 288):
Top 5 Commodities (ZSET ranking:week:94): [{'rank': 1, 'commodity_desc': 'COUPON/MISC ITEMS', 'week_sales': 7135.36}, {'rank': 2, 'commodity_desc': 'BEEF', 'week_sales': 3950.83}, {'rank': 3, 'commodity_desc': 'SOFT DRINKS', 'week_sales': 2904.12}, {'rank': 4, 'commodity_desc': 'FLUID MILK PRODUCTS', 'week_sales': 2664.05}, {'rank': 5, 'commodity_desc': 'CHEESE', 'week_sales': 2414.03}]
Total de produtos em display físico na loja 288 (SET store:288:week:94:displays): 100
